# Using KnowRob in Python

This notebook demonstrates how to use the KnowRob system directly in Python. It includes importing necessary modules, initializing the knowledge base, and executing queries.

### Importing KnowRob Modules

In [2]:
import json
from knowrob import *

First, we import the required modules from KnowRob. The `try-except` block ensures compatibility with different ROS environments, either using the ROS1-specific package or directly loading `knowrob.so`.

In [3]:
InitKnowRob()

[14:02:56.641] [info] [KnowRob] static initialization done.


The `InitKnowRob()` function initializes the KnowRob system, setting up necessary configurations and connections.

### Setting Up Knowledge Base

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			# {"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"}
            {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
		]
	},
	"data-sources": [
		# {"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
      "name": "pl",
      "type": "Prolog:rdf_db"
        }
	],
	"reasoner": [
        {
          "name": "pl",
          "type": "Prolog",
          "data-backend": "pl",
          "imports": [
            {
            "path": "tests/prolog/lpn.pl",
            "format": "prolog"
        }
      ]
    }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

In [3]:
# # Sample dictionary to be converted to JSON
# sample_dict = {
# 	"logging": {
# 		"console-sink": {"level": "debug"},
# 		"file-sink": {"level": "debug"}
# 	},
# 	"semantic-web": {
# 		"prefixes": [
# 			# {"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"}
#             {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
# 		]
# 	},
# 	"data-sources": [
# 		# {"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
#         {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
# 	],
# 	"data-backends": [
# 		{
# 			"type": "MongoDB",
# 			"name": "mongodb",
# 			"host": "localhost",
# 			"port": 27017,
# 			"db": "pizza",
# 			"read-only": False
# 		}
# 	],
# 	"reasoner": [
#     ]
# }
# # Convert the dictionary to a JSON string
# json_str = json.dumps(sample_dict)
# # Initialize the KnowledgeBase with the PropertyTree
# kb = KnowledgeBase(json_str)

[13:43:16.601] [info] Using backend `mongodb` with type `MongoDB`.
[13:43:16.601] [info] [mongodb] connected to mongodb://localhost:27017 (pizza.triples).
[13:43:16.605] [info] Using queryable backend with id 'mongodb'.
[13:43:16.605] [info] Using persistent backend with id 'mongodb'.
[13:43:16.638] [info] Loading ontology at '/home/malineni/ROS_WS/knowrob/tests/owl/pizza.owl' with version "Fri Mar 21 13:41:11 2025" and origin "pizza".


This block defines the configuration for the KnowledgeBase, including logging, semantic web prefixes, data sources, and backends. The configuration is then serialized to a JSON string and used to initialize the `KnowledgeBase` instance.

### Submitting a Query

In [4]:
# phi1 = QueryParser.parse("swrl_test:hasAncestor(swrl_test:'Fred', ?y)")
phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'AmericanSlicer', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'Mozarella', ?y)")

Here, a query is parsed using the `QueryParser`. The query checks for ancestors of the entity `Lea` within the `swrl_test` namespace.

### Retrieving Query Results

In [5]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

The query formulated in the previous step is submitted to the KnowledgeBase. The results are retrieved as a stream, and a queue is created to handle them.

### Processing Query Results


In [6]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

This block checks if the result is affirmative (`AnswerYes`) and prints each substitution found in the query result, listing variable bindings.

### Negative Query Result Handling


In [7]:
phi2 = QueryParser.parse("swrl_test:hasSibling(swrl_test:'Ernest', swrl_test:'Fred')")
resultStream = kb.submitQuery(phi2, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult2 = resultQueue.pop_front()
if isinstance(nextResult2, AnswerNo):
    print("result is negative")
else:
    print("result is positive")

result is positive


A second query checks for a specific condition, in this case, whether `Lea` is an ancestor of herself, which is expected to be false. The result is handled accordingly.

### Inconclusive Query Result Handling

In [9]:
phi3 = QueryParser.parse("r(?x, ?y)")
resultStream = kb.submitQuery(phi3, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult3 = resultQueue.pop_front()
if isinstance(nextResult3, AnswerDontKnow):
    print("We can't say if the result is true or false")

[10:21:18.550] [warning] Predicate r(?x, ?y) is neither materialized in the EDB nor defined by a reasoner.
We can't say if the result is true or false



The final example demonstrates handling a situation where the system cannot determine the truth value of the query, resulting in an `AnswerDontKnow` response.

In [ ]:
class LLMReasoner(RDFGoalReasoner):
    def __init__(self):
        super(LLMReasoner, self).__init__()
        self.np1 = IRIAtom("http://knowrob.org/kb/lpn#np1")
        self.defineRelation(IRIAtom("http://knowrob.org/kb/lpn#nr1"))

    def initializeReasoner(self, Reasoner, *args, **kwargs):
        return True

    def evaluate(self, goal: RDFGoal):
        pass


class LLMReasoner2(DataDrivenReasoner):
    def __init__(self):
        super(LLMReasoner2, self).__init__()
        self.np1 = IRIAtom("http://knowrob.org/kb/lpn#np1")
        self.np2 = IRIAtom("http://knowrob.org/kb/lpn#np2")